<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/geospatial/Practical%203%20Geospatial%20Data%20Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <span style="color:green">Kwanda Mazibuko - stdnr: 1077167
</span>

# <span style="color:blue">Imputation and Further Data Processing - Focusing on Tin (New Sn)
</span>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#ignoring warnings
import warnings
warnings.filterwarnings('ignore')

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

## Basic Data Processing

In [ ]:
url = "https://raw.githubusercontent.com/kwanda2426/projects/main/geospatial/Merged_Chem.csv"
df_1 = pd.read_csv(url)
df_1.head()

In [ ]:
df_1.columns

In [ ]:
# df preprocessing
df = df_1.copy()
df = df.drop(index = 0)
df = df.drop(['Unnamed: 0'], axis = 1)
df.head()

In [ ]:
# Fixing LOI
df['LOI'] = np.abs(pd.to_numeric(df['LOI'], errors='coerce'))


In [ ]:
# Next we will see how missing values affect our use. First, let's get a view of how much is missing.
df_stats = df.describe().T
df_stats.head()

In [ ]:
# We want to plot the count vs. index, but we should make the index an actual column first
df_stats = df_stats.reset_index(drop = False) #we're keeping the old index
df_stats = df_stats.rename(columns = dict([['index', 'column']]))
df_stats.head()

In [ ]:
# Let's create a list of elements again to facilitate this analysis
element_lst = [col for col in df.columns if ('Old') in col] + [col for col in df.columns if ('New') in col]

In [ ]:
# Element List and Fraction of missing values
count_lst = []
for element in element_lst:

    count_lst.append([element, (df[element].isna().sum()/len(df))])


count_df = pd.DataFrame(count_lst, columns = ['Element', 'Fraction Missing'])
# sorting the dataframe
count_df = count_df.sort_values(by = 'Fraction Missing')

# our element of interest has 15% ,missing values
count_df[count_df['Element'].isin(['New Sn'])]

In [ ]:
# Let's plot this
plt.figure(figsize = (16, 5))

plt.bar(count_df.Element, count_df['Fraction Missing'], color = 'grey')

plt.plot([len(count_df.Element), 0.1], [0.1, 0.1], color = 'black', label = '10% missing')
plt.plot([len(count_df.Element), 0.2], [0.2, 0.2], color = 'red', label = '20% missing')

plt.legend()
plt.xticks(rotation = 90)
plt.show()

- As seen that it is the first element above the 10% missing line. For elements to keep, we will include up to 16% to include our element of interest New Sn.

In [ ]:
# Let's build a list of elements that we won't keep for futher processing
discard_elements = [element for element in element_lst if df[element].isna().sum()/len(df)>=0.16]

In [ ]:
# Let's plot this
plt.figure(figsize = (16, 5))

plt.bar(count_df.Element, count_df['Fraction Missing'], color = 'green', label = 'Retained')

plt.bar(count_df[count_df.Element.isin(discard_elements)].Element,
        count_df[count_df.Element.isin(discard_elements)]['Fraction Missing'], color = 'red', label = 'Discarded')

plt.plot([len(count_df.Element), 0.16], [0.16, 0.16], color = 'black', label = '16% missing')
plt.plot([len(count_df.Element), 0.32], [0.32, 0.32], color = 'red', label = '32% missing')



plt.legend()
plt.xticks(rotation = 90)
plt.show()

- As seen that now we include New Sn which is our element of interest.

In [ ]:
# We now reconstruct the list to create an updated list of elements
retained_elems = [element for element in element_lst if element not in discard_elements]


In [ ]:
# we make our upper limit to 16% to include New Sn
demo_elements = [element for element in element_lst if (df[element].isna().sum()/len(df)>0.01) and (df[element].isna().sum()/len(df)<0.16)]
demo_elements

### Data Imputation

In [ ]:
# KNNImputer

from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors = 10)

In [ ]:
# Let's create a list of names for imputed columns
imputed_elems = ['Imputed '+elem for elem in retained_elems]
df[imputed_elems] = pd.DataFrame(imputer.fit_transform(df[retained_elems]))
df.describe()

In [ ]:
plt.figure(figsize = (8,8))


plt.scatter(df[retained_elems].mean(), df[imputed_elems].mean(),
            label = 'Mean', marker = 's', facecolor = 'none', edgecolor = 'black', s = 50)

plt.scatter(df[retained_elems].std(), df[imputed_elems].std(),
            label = 'STD', facecolor = 'none', edgecolor = 'red')

lim = max(max(df[retained_elems].mean()), max(df[imputed_elems].mean()), max(df[retained_elems].std()), max(df[imputed_elems].std()))


plt.plot([0,lim],[0,lim], linestyle = ':', color = 'k', label = 'Ideal')



plt.legend()
plt.xlabel('Original')
plt.ylabel('Imputed')
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import r2_score


r2_lst = []


for n in range(1, 100, 1):


    imputer = KNNImputer(n_neighbors=n, weights = 'distance')



    df[imputed_elems] = pd.DataFrame(imputer.fit_transform(df[retained_elems]), columns = retained_elems)



    r2_lst.append([n, r2_score(df[retained_elems].mean(), df[imputed_elems].mean()), r2_score(df[retained_elems].std(), df[imputed_elems].std())])

In [ ]:
r2_df = pd.DataFrame(r2_lst, columns = ['N', 'Mean', 'STD'])
plt.plot(r2_df.N, r2_df.STD, label = 'STD')
plt.plot(r2_df.N, r2_df.Mean, label = 'Mean')
plt.legend()


In [ ]:
plt.hist(df['Imputed New Sn'], bins = 100, alpha = 0.5, histtype = 'step', log = True, color = 'k')
plt.hist(df['New Sn'], bins = 100, alpha = 0.5, log = True, color = 'red')

In [ ]:
chemical = 'New Sn'

In [ ]:
r2_lst = []

for n in range(1, 100, 1):

    imputer = KNNImputer(n_neighbors=n, weights = 'distance')

    df[imputed_elems] = pd.DataFrame(imputer.fit_transform(df[retained_elems]), columns = retained_elems)

    imput_hist = plt.hist(df[imputed_elems[retained_elems.index(chemical)]], bins = 100, log = True)
    orig_hist = plt.hist(df[chemical].dropna(), bins = 100, range = (df[imputed_elems[retained_elems.index(chemical)]].min(), df[imputed_elems[retained_elems.index(chemical)]].max()), log = True)



    plt.close()
    r2_lst.append([n, r2_score(orig_hist[0], imput_hist[0])])


In [ ]:
r2_df = pd.DataFrame(r2_lst, columns = ['N', 'R2'])


plt.plot(r2_df.N, r2_df.R2)


In [ ]:
r2_df.R2.max()

In [ ]:
r2_df[r2_df.R2 == r2_df.R2.max()]

In [ ]:
imputer = KNNImputer(n_neighbors = 11, weights = 'distance')



df[imputed_elems] = pd.DataFrame(imputer.fit_transform(df[retained_elems]), columns = retained_elems)



df.describe()

In [ ]:
orig_hist[0]

In [ ]:
imput_hist = np.histogram(df[imputed_elems[retained_elems.index(chemical)]], bins = 100)
orig_hist = np.histogram(df[chemical].dropna(), bins = 100, range = (df[imputed_elems[retained_elems.index(chemical)]].min(), df[imputed_elems[retained_elems.index(chemical)]].max()))

lim = max(max(imput_hist[0]), max(orig_hist[0]))
plt.plot([0,lim],[0,lim], linestyle = ':', color = 'purple', label = 'Ideal')

plt.scatter(imput_hist[0], orig_hist[0])

In [ ]:
# Define the chemical you want to plot
chemical = 'New Sn'  # Replace with actual chemical name

# Get the index of the chemical in retained_elems
index = retained_elems.index(chemical)

# Set up the figure
fig_size = 6
fig, ax = plt.subplots(figsize=(fig_size, fig_size))

# Define colormaps
colormap1 = plt.cm.get_cmap('jet_r', len(retained_elems))
colormap2 = plt.cm.get_cmap('jet', len(retained_elems))

# Plot histograms
ax.hist(df[chemical], bins=100, color=colormap1(index),
        label=f'{chemical} Original', log=True)
ax.hist(df[imputed_elems[index]], bins=100, color=colormap2(index),
        label=f'{chemical} Imputed', log=True, histtype='step')

# Customize plot
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()


In [ ]:
imputed_demo_elements = ['Imputed '+element for element in demo_elements]
imputed_demo_elements

In [ ]:
for element in demo_elements:
    print(element, imputed_demo_elements[demo_elements.index(element)])
    temp_df = df[df[element].isna()][imputed_demo_elements[demo_elements.index(element)]]


    print(temp_df.mean())

In [ ]:
plt.hist(df['Imputed New Sn'], log = True)
plt.hist(df['New Sn'], histtype = 'step', log = True)

### Outlier Analysis

In [ ]:
df

## Element of Focus - Tin (Sn)

In [ ]:
# Let's take a look at one element - "Sn" for now

plt.hist(df['Imputed New Sn'], bins = 100, log = True) #enabling log helps, why?
plt.show()

### Assignment


#### Impute over the "New Sn" analysis. Optimize it using the distribution (histogram) + R2 metric based method. Then visualize your results the same way we did here for the demo elements.

What is the best number of neighbours for "New Sn"? Is it obvious?